# Half-year ACTIVSg2000 DCOPF benchmark

Run this notebook to generate, rather than assert, measurements for horizons 1, 24, 168, 720, 4368, and 4416 hours. No benchmark results are embedded.

## Setup

This installs the declared project dependencies and reporting helpers. The explicit highspy guard is required: if HiGHS cannot be imported, the notebook stops with an actionable error.

In [ ]:
import importlib.metadata
import os
from pathlib import Path
import platform
import subprocess
import sys

REPO_URL = os.environ.get("REPO_URL", "https://github.com/MhaoMou/ercot-connection-planning-instructor.git")
REPO_REF = os.environ.get("REPO_REF", "main")
REPO_DIR = Path(os.environ.get("REPO_DIR", "/content/ercot-connection-planning-instructor"))

if not (REPO_DIR / "pyproject.toml").is_file():
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not the expected repository")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "psutil", "pandas", "matplotlib"], check=True)
try:
    import highspy
except ImportError as exc:
    raise RuntimeError(
        "highspy is unavailable. Install the project dependencies in this cell and rerun; "
        "the benchmark cannot produce solver measurements without HiGHS."
    ) from exc

REPO_REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("repository:", REPO_DIR)
print("revision:", REPO_REVISION)
print("python:", sys.version)
print("platform:", platform.platform())
print("highspy:", importlib.metadata.version("highspy"))

## Configuration and public solver inputs

The checked-in scenario manifests are fail-closed until release validation is complete. The benchmark therefore materializes equivalent period inputs with the public deterministic profile functions and passes them to the existing low-level DCOPF solver. This changes no evaluator code or semantics.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
import json
import math
import os
import statistics
import time
import tracemalloc

import pandas as pd
import psutil

from tools.activsg2000 import dcopf as dcopf_module
from tools.activsg2000.dcopf import (
    Branch, Bus, Generator, StorageDispatchAsset,
    load_prepared_artifact, solve_dcopf, solve_dcopf_horizon,
)
from ercot_game.scenario_profiles import (
    block_offset, demand_multiplier, generator_availability, load_profile_inputs,
)

BENCHMARK_HORIZONS = (1, 24, 168, 720, 4368, 4416)
SHORT_HORIZON_CUTOFF = 720
REPEATS_SHORT = 3
REPEATS_LONG = 1
STORAGE_MAX_HORIZON = int(os.environ.get("STORAGE_MAX_HORIZON", "720"))
PARALLEL_WORKERS = max(2, min(4, os.cpu_count() or 2))
WALL_TIME_BUDGET_S = float(os.environ.get("WALL_TIME_BUDGET_S", "1800"))
PEAK_RSS_BUDGET_MB = float(os.environ.get("PEAK_RSS_BUDGET_MB", "12000"))
SCENARIO_ROOT = Path(os.environ.get(
    "ARTIFACT_ROOT", str(REPO_DIR / "data/processed/activsg2000_scenarios_v1")
))
ARTIFACT_4368 = SCENARIO_ROOT / "2028/jan_jun"
ARTIFACT_4416 = SCENARIO_ROOT / "2027/jul_dec"

def artifact_for(horizon):
    if horizon <= 4368:
        return ARTIFACT_4368
    if horizon <= 4416:
        return ARTIFACT_4416
    raise ValueError(f"unsupported horizon: {horizon}")

@lru_cache(maxsize=4)
def static_inputs(artifact_dir_string):
    root = Path(artifact_dir_string)
    artifact = load_prepared_artifact(root)
    if artifact.scenario_year is None or artifact.scenario_block is None:
        raise ValueError(f"{root} is missing scenario_profile.json")
    base_load, profiles, catalog = load_profile_inputs(root)
    generators = tuple(row for row in artifact.generators_raw if row.get("status", "1") == "1")
    branches = tuple(row for row in artifact.branches_raw
                     if row.get("status", "1") == "1" and float(row["reactance_pu"]) != 0.0)
    injections = {}
    for row in artifact.dcline_raw:
        if row.get("status", "1") == "1":
            injections[row["to_bus_id"]] = injections.get(row["to_bus_id"], 0.0) + float(row["Pt"])
    return artifact, base_load, profiles, catalog, generators, branches, injections

@lru_cache(maxsize=64)
def period_inputs(artifact_dir_string, period_index):
    artifact, base_load, profiles, catalog, generator_rows, branch_rows, injections = static_inputs(artifact_dir_string)
    absolute_hour = block_offset(artifact.scenario_year, artifact.scenario_block) + period_index
    multiplier = demand_multiplier(absolute_hour, year=artifact.scenario_year)
    demand = {bus_id: round(value * multiplier, 8) for bus_id, value in base_load.items()}
    availability = {
        row["generator_id"]: round(float(row["p_max_mw"]) * generator_availability(
            profiles[row["generator_id"]], absolute_hour, year=artifact.scenario_year
        ), 8)
        for row in catalog if int(row["status"]) == 1
    }
    buses = tuple(Bus(row["bus_id"], demand[row["bus_id"]], reference=row.get("type") == "3")
                  for row in artifact.buses_raw)
    generators = tuple(Generator(
        row["generator_id"], row["bus_id"], float(row["p_min_mw"]),
        min(float(row["p_max_mw"]), availability.get(row["generator_id"], float(row["p_max_mw"]))),
        float(row["marginal_cost_c1_per_mwh"])
    ) for row in generator_rows)
    branches = tuple(Branch(
        row["branch_id"], row["from_bus_id"], row["to_bus_id"],
        1.0 / float(row["reactance_pu"]), float(row["thermal_limit_mva"])
    ) for row in branch_rows)
    return buses, generators, branches, injections

def model_counts(artifact_dir, horizon, storage_assets=0, owner_groups=0):
    artifact, _base, _profiles, _catalog, generators, branches, _inj = static_inputs(str(artifact_dir))
    buses = len(artifact.buses_raw)
    gens = len(generators)
    lines = len(branches)
    network_vars = horizon * (gens + max(buses - 1, 0))
    return {
        "bus_count": buses, "generator_count": gens, "branch_count": lines,
        "network_variable_count": network_vars,
        "variable_count": network_vars + horizon * 4 * storage_assets,
        "constraint_count": horizon * (buses + lines + 4 * storage_assets + owner_groups),
    }

## Measurement helpers

serial clears the artifact/profile/topology caches before each period. sparse_reuse warms and reuses them. chunked_parallel uses deterministic threaded chunks, matching the existing worker pattern. Storage is one coupled horizon solve, so hourly P95 fields remain null.

In [ ]:
def clear_caches():
    static_inputs.cache_clear()
    period_inputs.cache_clear()
    dcopf_module.clear_prepared_artifact_cache()
    dcopf_module._cached_network_index.cache_clear()

def percentile(values, p):
    if not values:
        return None
    values = sorted(float(x) for x in values)
    if len(values) == 1:
        return values[0]
    position = (len(values) - 1) * p / 100.0
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return values[lo]
    return values[lo] + (values[hi] - values[lo]) * (position - lo)

def solve_one(root, index):
    buses, generators, branches, injections = period_inputs(str(root), index)
    return solve_dcopf(
        buses=buses, generators=generators, branches=branches,
        fixed_injection_mw_by_bus=injections, base_mva=100.0,
    )

def solve_serial(root, horizon, cold):
    if not cold:
        clear_caches()
    times, statuses = [], []
    for index in range(horizon):
        if cold:
            clear_caches()
        started = time.perf_counter()
        result = solve_one(root, index)
        times.append(time.perf_counter() - started)
        statuses.append(result.status)
    return {"hourly_times": times, "statuses": statuses}

def solve_chunked_parallel(root, horizon, workers):
    clear_caches()
    chunk_size = math.ceil(horizon / workers)
    chunks = [range(start, min(start + chunk_size, horizon))
              for start in range(0, horizon, chunk_size)]
    def run_chunk(chunk):
        times, statuses = [], []
        for index in chunk:
            started = time.perf_counter()
            result = solve_one(root, index)
            times.append(time.perf_counter() - started)
            statuses.append(result.status)
        return times, statuses
    with ThreadPoolExecutor(max_workers=workers) as executor:
        pieces = list(executor.map(run_chunk, chunks))
    return {
        "hourly_times": [t for times, _ in pieces for t in times],
        "statuses": [s for _, statuses in pieces for s in statuses],
    }

def solve_storage(root, horizon):
    clear_caches()
    periods, injections, owner_load = [], [], []
    for index in range(horizon):
        buses, generators, branches, fixed = period_inputs(str(root), index)
        periods.append((buses, generators, branches))
        injections.append(fixed)
        owner_load.append({buses[0].bus_id: max(buses[0].demand_mw, 0.0)})
    storage = StorageDispatchAsset(
        storage_id="benchmark-storage", team_id="benchmark-team",
        bus_id=periods[0][0][0].bus_id, power_mw=1.0,
        energy_mwh=4.0, initial_soc_mwh=2.0,
    )
    result = solve_dcopf_horizon(
        periods=tuple(periods), storage_assets=(storage,),
        fixed_injection_mw_by_period=tuple(injections),
        owner_load_mw_by_team={"benchmark-team": tuple(owner_load)},
        time_step_hours=1.0, base_mva=100.0,
    )
    return {"hourly_times": None, "statuses": [result.status]}

def timed(function):
    before = psutil.Process().memory_info().rss / (1024 * 1024)
    tracemalloc.start()
    started = time.perf_counter()
    payload, error = None, None
    try:
        payload = function()
    except Exception as exc:
        error = {"type": type(exc).__name__, "message": str(exc)}
    wall = time.perf_counter() - started
    _current, peak_python = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    after = psutil.Process().memory_info().rss / (1024 * 1024)
    return {
        "payload": payload, "error": error, "wall_time_s": wall,
        "peak_memory_mb": max(before, after),
        "python_peak_memory_mb": peak_python / (1024 * 1024),
        "process_rss_before_mb": before,
    }

def status_text(statuses):
    unique = sorted(set(statuses))
    return "not_run" if not unique else unique[0] if len(unique) == 1 else "|".join(unique)

## Execute all required cases

Results are written only after the cells actually run. Storage horizons above the cap receive explicit skipped rows with a reason.

In [ ]:
def row_for(horizon, root, variant, repeat_index, workers, counts, measurement, storage):
    payload = measurement["payload"] or {}
    error = measurement["error"]
    statuses = payload.get("statuses", [])
    run_status = "completed" if error is None else (
        "infeasible" if "Infeasible" in error["type"] else "error"
    )
    times = payload.get("hourly_times")
    return {
        "horizon_hours": horizon, "artifact_dir": str(root), "variant": variant,
        "storage_variant": "storage" if storage else "no_storage",
        "repeat_index": repeat_index, "worker_count": workers,
        "run_status": run_status, "solver_status": status_text(statuses),
        "wall_time_s": measurement["wall_time_s"],
        "median_hourly_solve_time_s": statistics.median(times) if times else None,
        "p95_hourly_solve_time_s": percentile(times, 95) if times else None,
        "peak_memory_mb": measurement["peak_memory_mb"],
        "python_peak_memory_mb": measurement["python_peak_memory_mb"],
        "process_rss_before_mb": measurement["process_rss_before_mb"],
        **counts,
        "error_type": error["type"] if error else None,
        "error_message": error["message"] if error else None,
    }

def skipped(horizon, root, repeat_index):
    counts = model_counts(root, horizon, storage_assets=1, owner_groups=1)
    return {
        "horizon_hours": horizon, "artifact_dir": str(root),
        "variant": "storage_horizon", "storage_variant": "storage",
        "repeat_index": repeat_index, "worker_count": 1,
        "run_status": "skipped", "solver_status": "not_run",
        "wall_time_s": None, "median_hourly_solve_time_s": None,
        "p95_hourly_solve_time_s": None, "peak_memory_mb": None,
        "python_peak_memory_mb": None, "process_rss_before_mb": None,
        **counts, "error_type": "SkippedByConfiguration",
        "error_message": f"storage horizon exceeds STORAGE_MAX_HORIZON={STORAGE_MAX_HORIZON}",
    }

def run_case(horizon, variant, repeat_index):
    root = artifact_for(horizon)
    if variant == "storage_horizon" and horizon > STORAGE_MAX_HORIZON:
        return skipped(horizon, root, repeat_index)
    storage = variant == "storage_horizon"
    counts = model_counts(root, horizon, storage_assets=int(storage), owner_groups=int(storage))
    if variant == "serial":
        function, workers = lambda: solve_serial(root, horizon, cold=True), 1
    elif variant == "sparse_reuse":
        function, workers = lambda: solve_serial(root, horizon, cold=False), 1
    elif variant == "chunked_parallel":
        function, workers = lambda: solve_chunked_parallel(root, horizon, PARALLEL_WORKERS), PARALLEL_WORKERS
    elif variant == "storage_horizon":
        function, workers = lambda: solve_storage(root, horizon), 1
    else:
        raise ValueError(variant)
    return row_for(horizon, root, variant, repeat_index, workers, counts, timed(function), storage)

def add_repeat_stats(rows):
    groups = {}
    for row in rows:
        if row["run_status"] == "completed" and row["wall_time_s"] is not None:
            groups.setdefault((row["horizon_hours"], row["variant"]), []).append(row["wall_time_s"])
    for row in rows:
        values = groups.get((row["horizon_hours"], row["variant"]), [])
        row["repeat_count_completed"] = len(values)
        row["median_wall_time_s"] = statistics.median(values) if values else None
        row["p95_wall_time_s"] = percentile(values, 95) if values else None
    return rows

def run_benchmark():
    rows = []
    for horizon in BENCHMARK_HORIZONS:
        repeats = REPEATS_SHORT if horizon <= SHORT_HORIZON_CUTOFF else REPEATS_LONG
        for variant in ("serial", "sparse_reuse", "chunked_parallel", "storage_horizon"):
            for repeat_index in range(1, repeats + 1):
                print(f"running horizon={horizon} variant={variant} repeat={repeat_index}/{repeats}")
                rows.append(run_case(horizon, variant, repeat_index))
    return add_repeat_stats(rows)

def recommend(rows):
    candidates = [row for row in rows if row["horizon_hours"] == 4416
                  and row["variant"] == "sparse_reuse"
                  and row["run_status"] == "completed"]
    if not candidates:
        return {"recommendation": "no_recommendation",
                "reason": "No completed 4416-hour sparse_reuse measurement exists."}
    wall = max(row["median_wall_time_s"] for row in candidates)
    memory = max(row["peak_memory_mb"] for row in candidates)
    if wall <= WALL_TIME_BUDGET_S and memory <= PEAK_RSS_BUDGET_MB:
        decision = "retain_official_4416_hour_horizon"
        reason = "Measured full-horizon sparse_reuse results are within configured budgets."
    else:
        decision = "screen_or_shrink_locally_only"
        reason = ("Measured full-horizon results exceed a configured budget. "
                  "Keep full remote instructor evaluation; shorter runs are screening-only.")
    return {"recommendation": decision, "reason": reason,
            "median_wall_time_s": wall, "peak_memory_mb": memory,
            "wall_time_budget_s": WALL_TIME_BUDGET_S,
            "peak_rss_budget_mb": PEAK_RSS_BUDGET_MB}

In [ ]:
results = run_benchmark()
results_frame = pd.DataFrame(results)
output_dir = REPO_DIR / "experiments/colab/results"
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "dcopf_benchmark_results.csv"
json_path = output_dir / "dcopf_benchmark_results.json"
results_frame.to_csv(csv_path, index=False)
json_path.write_text(json.dumps({
    "repository_revision": REPO_REVISION,
    "python": sys.version,
    "platform": platform.platform(),
    "highspy_version": importlib.metadata.version("highspy"),
    "configuration": {
        "horizons": list(BENCHMARK_HORIZONS), "repeats_short": REPEATS_SHORT,
        "repeats_long": REPEATS_LONG, "storage_max_horizon": STORAGE_MAX_HORIZON,
        "parallel_workers": PARALLEL_WORKERS,
        "wall_time_budget_s": WALL_TIME_BUDGET_S,
        "peak_rss_budget_mb": PEAK_RSS_BUDGET_MB,
    },
    "rows": results, "recommendation": recommend(results),
}, indent=2), encoding="utf-8")
display(results_frame[[
    "horizon_hours", "variant", "storage_variant", "repeat_index",
    "run_status", "solver_status", "wall_time_s",
    "median_hourly_solve_time_s", "p95_hourly_solve_time_s",
    "peak_memory_mb", "worker_count", "network_variable_count",
    "variable_count", "constraint_count",
]])
print("wrote:", csv_path)
print("wrote:", json_path)
print(json.dumps(recommend(results), indent=2))

## Interpretation

The recommendation is generated only from completed 4,416-hour no-storage sparse_reuse measurements and configured wall-time/RSS budgets. If no such row exists, the notebook reports no recommendation. A student-laptop shorter run is a screening recommendation only; it must not change official horizon or evaluator semantics.

In [ ]:
import matplotlib.pyplot as plt

completed = results_frame[results_frame["run_status"] == "completed"]
if completed.empty:
    print("No completed rows to plot.")
else:
    for variant, group in completed.groupby("variant"):
        plt.plot(group["horizon_hours"], group["wall_time_s"], marker="o", label=variant)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Horizon (hours)")
    plt.ylabel("Wall time (seconds)")
    plt.title("Measured DCOPF benchmark wall time")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.show()